In [98]:
import torch
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
from tqdm.notebook import tqdm

In [99]:
vae = model.AutoEncoder({})

In [100]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])

In [116]:
class EpisodicDataset(torch.utils.data.Dataset):
    def __init__(self, cfg):
        super().__init__()
        self.db = lancedb.connect(cfg.dataset.lancedb_uri)
        self.table = self.db.open_table("episodes")

    def __getitem__(self, idx):
        episode_id = idx//self.table.count_rows()
        episode = self.table.search().where(f'episode_id = {idx}').limit(1).to_arrow()
        obs = episode['observations']
        print(obs)
        obs_array = np.array(obs, dtype=np.uint8).reshape(-1, 96, 96, 3)
        chunks = obs.combine_chunks()
        print(obs_array.shape)
        t = torch.from_numpy(np.stack(chunks.to_pylist())).to(torch.uint8)
        print(t.shape)
        return F.interpolate(t.view(-1, 96, 96, 3).permute(0, 3, 1, 2), (64, 64), mode="bilinear", align_corners=False).to(torch.float32)/255
    
    def __len__(self):
        return self.table.count_rows()

In [117]:
ds = EpisodicDataset(cfg)
train_ds, valid_ds = random_split(ds, [0.9, 0.1])

In [118]:
train_loader = DataLoader(train_ds, batch_size=1)
val_loader = DataLoader(valid_ds, batch_size=1)

In [119]:
train_ds[2]

KeyError: 'Field "observation" does not exist in schema'

In [112]:
optimizer = torch.optim.AdamW(vae.parameters())
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.training.nb_epochs)

for _ in tqdm(range(cfg.training.nb_epochs)):
    for batch in tqdm(train_loader):
        loss = F.mse_loss(vae(batch),x)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/260 [00:00<?, ?it/s]

[

]
(0, 96, 96, 3)


ValueError: need at least one array to stack